# Feature engineering from anonymised columns

This experiment tests whether label-free transformations of the anonymised transaction features improve temporal generalisation. Since the meanings of the 165 `feat_*` columns are unknown, the transformations describe relative positions and overall feature distributions rather than assuming domain-specific interpretations.

| Set | Features |
|---|---|
| A | Original numerical features |
| R | Original features plus within-time-step percentile ranks |
| Ronly | Within-time-step percentile ranks only |
| S | Original features plus row-level summary statistics |
| RS | Original features, ranks and row summaries |

Percentile ranks use the complete contemporaneous batch without target labels. This is a batch-transductive representation, which may not be available for real-time prediction.

Five chronological validation splits compare the feature sets using a rank-average of HistGradientBoosting and LightGBM predictions. **ROC-AUC is the primary selection metric.** A feature set is adopted only if its mean ROC-AUC exceeds the raw-feature baseline by at least 0.001 and it improves at least four of the five splits.


In [1]:
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

FAST = os.environ.get('BT_FAST', '0') == '1'      # BT_FAST=1 only for a quick smoke test
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'raw'
OUT = ROOT / 'submissions'; OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA / 'train.csv'); test = pd.read_csv(DATA / 'test.csv'); sample = pd.read_csv(DATA / 'sample_submission.csv')
featA = [c for c in train.columns if c.startswith('feat_')]
assert all(c in test.columns for c in featA)
assert 'time_step' in test.columns, 'test.csv needs time_step for within-step ranks'
allx = pd.concat([train.assign(is_test=0), test.assign(is_test=1)], ignore_index=True)
allx['label'] = allx['label'].astype(float)
is_lab = (allx.is_test == 0) & allx.label.notna()
print('Labelled:', int(is_lab.sum()), '| test:', int((allx.is_test == 1).sum()), '| features:', len(featA))
print('Rows per time step in test (min/median/max):', test.groupby('time_step').size().agg(['min', 'median', 'max']).tolist())

Labelled: 31235 | test: 15329 | features: 165
Rows per time step in test (min/median/max): [471.0, 1157.5, 2154.0]


## Feature representations

Construct percentile ranks within each time step and row-level summary statistics, then assemble the five feature sets for comparison.


In [2]:
rank_df = allx.groupby('time_step')[featA].rank(pct=True)
rank_df.columns = ['r_' + c for c in featA]
X = allx[featA].to_numpy(float)
z = (X - np.nanmean(X, 0)) / (np.nanstd(X, 0) + 1e-9)
summ = pd.DataFrame({'s_mean': np.nanmean(X, 1), 's_std': np.nanstd(X, 1), 's_min': np.nanmin(X, 1), 's_max': np.nanmax(X, 1),
                     's_n_extreme': (np.abs(z) > 3).sum(1)}, index=allx.index)
allx = pd.concat([allx, rank_df, summ], axis=1)
rcols, scols = list(rank_df.columns), list(summ.columns)
fsets = {'A': featA, 'R': featA + rcols, 'Ronly': rcols, 'S': featA + scols, 'RS': featA + rcols + scols}
print({k: len(v) for k, v in fsets.items()})

{'A': 165, 'R': 330, 'Ronly': 165, 'S': 170, 'RS': 335}


## Chronological evaluation

The first two folds validate immediately after the training period; the remaining folds leave a temporal gap to test robustness to more distant data. Performance can vary substantially with the training cutoff, so selection considers results across all five folds rather than a single period.


In [3]:
HGB_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=42)
LGB_PARAMS = dict(n_estimators=400, learning_rate=0.045, num_leaves=31, max_depth=-1, reg_lambda=5.0, random_state=42, verbosity=-1, n_jobs=-1)
if FAST: HGB_PARAMS['max_iter'] = 20; LGB_PARAMS['n_estimators'] = 20
FOLDS = [(24, 25, 29), (29, 30, 35), (19, 25, 29), (24, 30, 35), (19, 30, 35)]
rk = lambda p: rankdata(p) / len(p)

rows = []
for name, cols in fsets.items():
    aucs = []
    for tmax, lo, hi in FOLDS:
        tr = allx[is_lab & (allx.time_step <= tmax)]; va = allx[is_lab & allx.time_step.between(lo, hi)]
        ytr, yva = tr.label.astype(int), va.label.astype(int)
        ph = HistGradientBoostingClassifier(**HGB_PARAMS).fit(tr[cols], ytr).predict_proba(va[cols])[:, 1]
        pl = LGBMClassifier(**LGB_PARAMS).fit(tr[cols], ytr).predict_proba(va[cols])[:, 1]
        aucs.append(roc_auc_score(yva, 0.5 * rk(ph) + 0.5 * rk(pl)))
    rows.append(dict(featset=name, n_feat=len(cols), **{f'{lo}-{hi} (<={t})': a for (t, lo, hi), a in zip(FOLDS, aucs)}, mean_auc=np.mean(aucs)))
    print(name, 'done', round(np.mean(aucs), 5), flush=True)
res = pd.DataFrame(rows).set_index('featset')
fold_cols = [c for c in res.columns if c not in ('n_feat', 'mean_auc')]
display(res.round(5))

A done 0.93853
R done 0.93779
Ronly done 0.8752
S done 0.93332
RS done 0.92767


,n_feat,25-29 (<=24),30-35 (<=29),25-29 (<=19),30-35 (<=24),30-35 (<=19),mean_auc
featset,,,,,,,
A,165,0.99714,0.99756,0.84087,0.99592,0.86119,0.93853
R,330,0.99635,0.99762,0.82863,0.99483,0.87153,0.93779
Ronly,165,0.97922,0.87668,0.83779,0.85486,0.82744,0.87520
S,170,0.99654,0.99753,0.81121,0.99496,0.86635,0.93332
RS,335,0.99645,0.99715,0.79314,0.99400,0.85760,0.92767


## Feature selection

Compare each engineered feature set with the original columns using the prespecified mean ROC-AUC gain and number of improved folds.


In [4]:
delta = res[fold_cols].sub(res.loc['A', fold_cols], axis=1)
summary = pd.DataFrame({'mean_gain_vs_A': res.mean_auc - res.loc['A', 'mean_auc'], 'folds_improved_of_5': (delta > 0).sum(axis=1)})
display(summary.round(5))
cands = summary.drop(index='A')
ok = cands[(cands.mean_gain_vs_A >= 0.001) & (cands.folds_improved_of_5 >= 4)]
CHOSEN = ok.mean_gain_vs_A.idxmax() if len(ok) else 'A'
print('Adopted feature set:', CHOSEN, '->', 'NEW SET PASSES the rule' if CHOSEN != 'A' else 'no set passes; keep Exp18 unchanged')

,mean_gain_vs_A,folds_improved_of_5
featset,,
A,0.00000,0
R,-0.00074,2
Ronly,-0.06334,0
S,-0.00522,1
RS,-0.01087,0


Adopted feature set: A -> no set passes; keep Exp18 unchanged


## Chronological stacking check

If a feature set meets the selection rule, evaluate it with the existing HGB–CatBoost–LightGBM chronological stacking procedure. The meta-model is trained on earlier out-of-fold predictions, followed by evaluation on steps 25–29 and 30–35. Submission generation is conditional on the selected feature set; the original stack remains the reference model.


In [5]:
FORCE_RUN = False
cols = fsets[CHOSEN]
if CHOSEN == 'A' and not FORCE_RUN:
    print('Skipped: baseline features kept. Exp18 remains your main model.')
else:
    SEEDS = [11, 42, 73, 101, 202]
    CAT_PARAMS = dict(iterations=700, depth=6, learning_rate=0.045, l2_leaf_reg=5, loss_function='Logloss', verbose=False, thread_count=-1)
    LGB18 = dict(n_estimators=700, learning_rate=0.045, num_leaves=31, max_depth=-1, reg_lambda=5.0, random_state=42, verbosity=-1, n_jobs=-1)
    if FAST: SEEDS = [11]; CAT_PARAMS['iterations'] = 20; LGB18['n_estimators'] = 20
    def base_preds(tr, va):
        y = tr.label.astype(int)
        ph = HistGradientBoostingClassifier(**HGB_PARAMS).fit(tr[cols], y).predict_proba(va[cols])[:, 1]
        pc = np.mean([CatBoostClassifier(**CAT_PARAMS, random_seed=s).fit(tr[cols], y).predict_proba(va[cols])[:, 1] for s in SEEDS], axis=0)
        pl = LGBMClassifier(**LGB18).fit(tr[cols], y).predict_proba(va[cols])[:, 1]
        return np.column_stack([ph, pc, pl])
    lab = allx[is_lab]
    Pm, ym = [], []
    for lo, hi in [(10, 14), (15, 19), (20, 24)]:
        tr, va = lab[lab.time_step < lo], lab[lab.time_step.between(lo, hi)]
        Pm.append(base_preds(tr, va)); ym.append(va.label.astype(int).to_numpy()); print('meta window', lo, hi, 'done', flush=True)
    stacker = LogisticRegression(C=0.1, max_iter=2000).fit(np.vstack(Pm), np.concatenate(ym))
    print('Coefficients [HGB, CatBoost, LightGBM]:', stacker.coef_.round(3), 'intercept', stacker.intercept_.round(3))
    exp18_ref = {'25-29': 0.997780, '30-35': 0.998414}
    ev = []
    for lo, hi in [(25, 29), (30, 35)]:
        tr, va = lab[lab.time_step < lo], lab[lab.time_step.between(lo, hi)]
        y = va.label.astype(int).to_numpy(); ps = stacker.predict_proba(base_preds(tr, va))[:, 1]
        ev.append(dict(period=f'{lo}-{hi}', new_roc=roc_auc_score(y, ps), exp18_roc=exp18_ref[f'{lo}-{hi}'], new_pr=average_precision_score(y, ps)))
    ev = pd.DataFrame(ev); ev['delta_roc'] = ev.new_roc - ev.exp18_roc; display(ev.round(6))
    te = allx[allx.is_test == 1]
    pred = stacker.predict_proba(base_preds(lab, te))[:, 1]
    if 'index' in test.columns: assert (test['index'].to_numpy() == sample['index'].to_numpy()).all()
    assert len(pred) == len(sample) and sample.columns.tolist() == ['index', 'target'] and np.isfinite(pred).all()
    sub = sample.copy(); sub['target'] = pred
    path = OUT / 'experiment20_ranked_stack.csv'; i = 2
    while path.exists(): path = OUT / f'experiment20_ranked_stack_v{i}.csv'; i += 1
    sub.to_csv(path, index=False); print('Saved:', path)
    print('Submit only if delta_roc is positive on BOTH windows; otherwise keep Exp18 + the blend as your finals.')

Skipped: baseline features kept. Exp18 remains your main model.


## Findings

The saved ROC-AUC evaluation did not identify an engineered feature set that satisfied the adoption rule. The original features therefore remain the selected representation, and the optional stacking stage is skipped. Within-period ranks and row summaries did not consistently improve chronological validation under the tested settings.

These results apply to the tested transformations and model configurations; they do not establish that anonymised features cannot support other useful representations.
